# Scientific metadata with Ontolutils

Run these cells in order after `python -m pip install ontolutils`. This notebook creates a model, checks a value, and saves and reloads RDF. Files are written in the notebook's working directory.

In [ ]:
from typing import Optional
from pydantic import Field, ValidationError
from ontolutils import Thing, QUDT_UNIT, namespaces, urirefs
from ontolutils.typing import AnyIri

@namespaces(ex="https://example.org/vocab/")
@urirefs(Measurement="ex:Measurement", value="ex:value", unit="ex:unit", note="ex:note")
class Measurement(Thing):
    value: float = Field(ge=0, alias="measured_value")
    unit: AnyIri
    note: Optional[str] = None

measurement = Measurement(
    id="https://example.org/measurements/1",
    measured_value=1.5,
    unit=str(QUDT_UNIT.M_PER_SEC),
    label="Velocity magnitude",
)
assert measurement.value == 1.5

`value` and `unit` are required. `note` can be omitted or set to `None`. The alias `measured_value` is an alternative Python input name; the RDF predicate remains `ex:value`. The example vocabulary should be replaced by your community's terms.

In [ ]:
try:
    measurement.value = -1
except ValidationError as error:
    print(error.errors()[0]["loc"])
else:
    raise AssertionError("Expected a validation error")
assert measurement.value == 1.5

## Save and reload

JSON-LD is JSON with RDF context. `model_dump_jsonld()` returns text; `get_jsonld_dict()` returns a dictionary. A loader normally returns a list; use `limit=1` for a single match.

In [ ]:
from pathlib import Path

document = measurement.model_dump_jsonld()
Path("measurement.jsonld").write_text(document, encoding="utf-8")
restored = Measurement.from_jsonld(source="measurement.jsonld", limit=1)
assert restored.value == 1.5
assert str(restored.unit) == str(QUDT_UNIT.M_PER_SEC)
print(measurement.serialize("ttl"))

## Inspect the RDF graph

Compare triples rather than serializer whitespace or prefix order. Keep stable IDs when sharing metadata across files.

In [ ]:
from rdflib import Graph, URIRef
from rdflib.compare import isomorphic

original_graph = Graph().parse(data=document, format="json-ld")
restored_graph = Graph().parse(data=restored.serialize("ttl"), format="ttl")
assert isomorphic(original_graph, restored_graph)
assert float(original_graph.value(URIRef(measurement.id), URIRef(Measurement.iri("value")))) == 1.5

## Add provenance with a built-in model

An object includes its metadata in the graph. An IRI string refers to metadata defined elsewhere.

In [ ]:
from ontolutils.ex.prov import Person, Entity

researcher = Person(id="https://example.org/people/ada", first_name="Ada")
dataset = Entity(
    id="https://example.org/datasets/run-01",
    label="Velocity measurements",
    was_attributed_to=researcher,
)
assert dataset.wasAttributedTo is researcher
print(dataset.model_dump_jsonld())

Python validation checks the declared fields, not every rule of an ontology. Use SHACL for RDF requirements such as mandatory labels. See the [user guide](https://ontology-utils.readthedocs.io/en/latest/userguide/index.html) for numerical variables, configuration, and graph validation.